# ARGUS · Session 2 Continuation — Phase B Resume (epochs 5→20)

## Before starting

### Step 1 — Extract from `_output_ .zip` (already on your Mac)
```
runs/s2_phb/argus_s2_phb/weights/last.pt   (474 MB)
runs/s2_phb/argus_s2_phb/args.yaml         (2 KB)
```

### Step 2 — Upload as a Kaggle dataset
1. Go to kaggle.com → Datasets → New Dataset
2. Name it exactly: `argus-s2-cont`
3. Upload both files (`last.pt` and `args.yaml`) — keep original filenames
4. Create the dataset

### Step 3 — Attach datasets to this notebook
Same as Session 2:
- `abhishekprajapat/idd-20k`
- BDD100K (bratjay)
- UA-DETRAC (marquis03)
- Your `argus-s2-cont` dataset (the one you just created)

### What this notebook does
Resumes Phase B from epoch 4. Each restart (when the 9.8h session limit hits)
auto-picks up from last.pt. Run until all 16h of quota are consumed:
- Session 1: epochs 5–7 (~9.3h training + 0.5h dataset build)
- Session 2: epochs 8–9 (~5.7h training + 0.5h dataset build)

Total Phase B after this week: 9/20 epochs → feeds into S2.5 next week.


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
from pathlib import Path
import torch, os, json as _json, shutil

NC      = 5
CLASSES = ['car', 'motorcycle', 'bus', 'truck', 'bicycle']

WORK     = Path('/kaggle/working')
OUT_DIR  = WORK / 'argus_data'
MERGED   = OUT_DIR / 'merged'
RUNS_DIR = WORK / 'runs'
HELD_DIR = OUT_DIR / 'held'
INPUT    = Path('/kaggle/input')
yaml_path = MERGED / 'data.yaml'

for d in [MERGED/'train/images', MERGED/'train/labels',
          MERGED/'valid/images', MERGED/'valid/labels',
          HELD_DIR/'valid/images', HELD_DIR/'valid/labels',
          RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Phase B run paths — must match Session 2 exactly for resume to work
PHB_DIR  = RUNS_DIR / 's2_phb' / 'argus_s2_phb'
PHB_LAST = PHB_DIR / 'weights' / 'last.pt'
PHB_BEST = PHB_DIR / 'weights' / 'best.pt'
(PHB_DIR / 'weights').mkdir(parents=True, exist_ok=True)

DEVICE = ','.join(str(i) for i in range(torch.cuda.device_count())) or 'cpu'
BATCH  = 16
IMGSZ  = 640

KGL_USER = os.environ.get('KAGGLE_USERNAME', '')
KGL_KEY  = os.environ.get('KAGGLE_KEY', '')
if KGL_USER and KGL_KEY:
    kdir = Path.home()/'.kaggle'; kdir.mkdir(exist_ok=True)
    (kdir/'kaggle.json').write_text(_json.dumps({'username':KGL_USER,'key':KGL_KEY}))
    (kdir/'kaggle.json').chmod(0o600)
    print(f'Kaggle creds: {KGL_USER}')
else:
    print('Kaggle creds NOT set — using attached datasets only')

print(f'DEVICE={DEVICE}  BATCH={BATCH}  IMGSZ={IMGSZ}')


In [ ]:
# ── Kaggle API credentials ────────────────────────────────────────────────────
import os, json as _j
from pathlib import Path

KAGGLE_USER = ''   # <- paste your Kaggle username
KAGGLE_KEY  = ''   # <- paste your API key

if not KAGGLE_USER: KAGGLE_USER = os.environ.get('KAGGLE_USERNAME', '')
if not KAGGLE_KEY:  KAGGLE_KEY  = os.environ.get('KAGGLE_KEY', '')
if KAGGLE_USER and KAGGLE_KEY:
    kdir = Path.home()/'.kaggle'; kdir.mkdir(exist_ok=True)
    (kdir/'kaggle.json').write_text(_j.dumps({'username':KAGGLE_USER,'key':KAGGLE_KEY}))
    (kdir/'kaggle.json').chmod(0o600)
    os.environ['KAGGLE_USERNAME'] = KAGGLE_USER
    os.environ['KAGGLE_KEY']      = KAGGLE_KEY
    print(f'Kaggle: authenticated as {KAGGLE_USER}')
else:
    print('Kaggle: no credentials — attached datasets only')


In [ ]:
# ── Install + GPU check ──────────────────────────────────────────────────────
import subprocess, sys, torch

n_gpu = torch.cuda.device_count()
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | GPUs: {n_gpu}')
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory//1024**2} MB')

subprocess.run([sys.executable,'-m','pip','install','-q',
                'ultralytics>=8.4.0','pyyaml','pycocotools'], check=True)
import ultralytics; ultralytics.checks()


In [ ]:
# ── Locate and reconstruct Phase B checkpoint for resume ──────────────────────
# Searches for last.pt + args.yaml from the argus-s2-cont Kaggle dataset.
# Reconstructs /kaggle/working/runs/s2_phb/argus_s2_phb/ so YOLO resume works.
import shutil
from pathlib import Path

SLUGS = ['argus-s2-weights', 'argus-s2-cont', 'argus-s2-checkpoint', 'argus_s2_cont']

checkpoint_dir = None
for s in SLUGS:
    if (INPUT/s).exists():
        checkpoint_dir = INPUT/s; break
if checkpoint_dir is None:
    # Nested datasets/<user>/<ds>/ search
    hit = next(INPUT.rglob('args.yaml'), None)
    if hit and (hit.parent/'last.pt').exists():
        checkpoint_dir = hit.parent
    else:
        hit2 = next(INPUT.rglob('last.pt'), None)
        if hit2: checkpoint_dir = hit2.parent

if checkpoint_dir is None:
    raise FileNotFoundError(
        'argus-s2-cont dataset not found.\n'
        'Create a Kaggle dataset named argus-s2-cont with last.pt + args.yaml '
        'extracted from _output_ .zip, then attach it to this notebook.')

print(f'Checkpoint dir: {checkpoint_dir}')

src_last = next((checkpoint_dir/n for n in ['last.pt','argus_s2_last.pt']
                 if (checkpoint_dir/n).exists()), None)
src_args = next((checkpoint_dir/n for n in ['args.yaml']
                 if (checkpoint_dir/n).exists()), None)

assert src_last, f'last.pt not found in {checkpoint_dir}'
print(f'  last.pt : {src_last}  ({src_last.stat().st_size//1024**2} MB)')

# Copy to exact path YOLO expects
if not PHB_LAST.exists():
    shutil.copy2(src_last, PHB_LAST)
    print(f'  Copied → {PHB_LAST}')
else:
    print(f'  last.pt already at {PHB_LAST}')

if src_args and not (PHB_DIR/'args.yaml').exists():
    shutil.copy2(src_args, PHB_DIR/'args.yaml')
    print(f'  Copied args.yaml → {PHB_DIR}/args.yaml')

print('\nCheckpoint ready — will resume from epoch 4')


In [ ]:
# ── Pre-flight: what's in /kaggle/input/ ──────────────────────────────────────
import os
from pathlib import Path

print('Mounted datasets in /kaggle/input/datasets/:')
ds_root = INPUT / 'datasets'
mounted_names = set()   # user dir names
dataset_names = set()   # actual dataset dir names
if ds_root.exists():
    for user_dir in sorted(ds_root.iterdir()):
        if not user_dir.is_dir(): continue
        print(f'  {user_dir.name}/')
        mounted_names.add(user_dir.name.lower())
        for ds_dir in sorted(user_dir.iterdir()):
            if not ds_dir.is_dir(): continue
            dataset_names.add(ds_dir.name.lower())
            try:
                mb = sum(f.stat().st_size for f in ds_dir.rglob('*') if f.is_file()) // 1024**2
                print(f'    {ds_dir.name}/  (~{mb} MB)')
            except Exception:
                print(f'    {ds_dir.name}/')
elif INPUT.exists():
    for d in sorted(INPUT.iterdir()):
        if d.is_dir():
            mounted_names.add(d.name.lower()); print(f'  {d.name}/')

all_names = mounted_names | dataset_names
print()
ckpt_ok = PHB_LAST.exists() or next(INPUT.rglob('last.pt'), None) is not None
idd_ok  = any('idd' in n for n in all_names) or next(INPUT.rglob('idd20k_final'), None) is not None
bdd_ok  = any(k in n for n in all_names for k in ['bdd','bd1']) or bool(KGL_USER)
print(f'  S2 checkpoint : {"OK" if ckpt_ok else "MISSING -- attach argus-s2-weights dataset"}')
print(f'  IDD dataset   : {"OK" if idd_ok  else "MISSING -- attach abhishekprajapat/idd-20k"}')
print(f'  BDD100K       : {"OK" if bdd_ok  else "MISSING -- attach a BDD dataset or set Kaggle creds"}')
print()
if not ckpt_ok:
    print('WARNING: S2 checkpoint missing — checkpoint-setup cell will fail')
elif not idd_ok:
    print('WARNING: IDD missing')
else:
    print('All required datasets found -- proceed')


In [ ]:
# ── UA-DETRAC → YOLO (overhead CCTV) ─────────────────────────────────────────
# Adds overhead traffic camera viewpoint — critical for near-miss scenarios.
# car→0  bus→2  van/truck→3  motorbike→1
# 2k held-out images locked per session (random.seed(42) — deterministic).
import subprocess, os, re, random, shutil as _shutil, xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm

def _imglink(src, dst):
    try: os.link(src, dst)
    except OSError: os.symlink(Path(src).resolve(), dst)

random.seed(42)
FLAG = OUT_DIR / '.uadetrac_done'
if FLAG.exists():
    print('UA-DETRAC: already done — skipping')
else:
    DETRAC_MAP = {'car':0,'van':3,'truck':3,'bus':2,'motorbike':1,'motorcycle':1,'others':0}
    IMG_W, IMG_H = 960, 540  # UA-DETRAC standard resolution

    DETRAC_ROOT = next((INPUT/s for s in ['ua-detrac','uadetrac','ua_detrac','detrac-training']
                        if (INPUT/s).exists()), None)
    # Also search nested datasets/<user>/<ds>/ (Kaggle merged dataset mounting)
    if DETRAC_ROOT is None:
        ds_root = INPUT / 'datasets'
        if ds_root.exists():
            for user_dir in sorted(ds_root.iterdir()):
                if not user_dir.is_dir() or DETRAC_ROOT: break
                for ds_dir in sorted(user_dir.iterdir()):
                    if not ds_dir.is_dir(): continue
                    # UA-DETRAC: XML files + images in MVI_* sequence dirs
                    xml_count = sum(1 for _ in ds_dir.rglob('*.xml'))
                    if xml_count > 50:
                        DETRAC_ROOT = ds_dir
                        print(f'UA-DETRAC: found at datasets/{user_dir.name}/{ds_dir.name}/ ({xml_count} XMLs)')
                        break
    if DETRAC_ROOT is None and KGL_USER:
        RAW = WORK/'_detrac_raw'; RAW.mkdir(parents=True, exist_ok=True)
        dl_flag = RAW/'.downloaded'
        SLUGS = ['datatang/ua-detrac','dtrackers/ua-detrac','fanbyprince/ua-detrac',
                 'robustchicken/ua-detrac','adamdodge/ua-detrac']
        if dl_flag.exists():
            print('UA-DETRAC: already downloaded'); DETRAC_ROOT = RAW
        else:
            for slug in SLUGS:
                r = subprocess.run(['kaggle','datasets','download','-d',slug,
                                    '-p',str(RAW),'--unzip'], capture_output=True, text=True)
                if r.returncode == 0:
                    dl_flag.touch(); DETRAC_ROOT = RAW
                    print(f'  Downloaded via {slug}'); break
                print(f'  {slug}: {r.stderr.strip()[:100]}')
            else:
                print('  All UA-DETRAC slugs failed — skipping')

    if DETRAC_ROOT is None:
        print('UA-DETRAC: not found — set Kaggle creds or attach dataset')
    else:
        # Build seq_name → XML map (UA-DETRAC: one XML per sequence, e.g. MVI_20011.xml)
        seq_xmls = sorted(DETRAC_ROOT.rglob('*.xml'))
        xml_by_seq = {x.stem: x for x in seq_xmls}
        print(f'Found {len(seq_xmls)} sequence XMLs')

        # Parse all XMLs into memory: seq_name → {frame_num → [(ac,cx,cy,nw,nh)]}
        seq_ann = {}
        for xp in tqdm(seq_xmls, desc='Parse XMLs'):
            try: root_el = ET.parse(xp).getroot()
            except ET.ParseError: continue
            frames = {}
            for frame in root_el.findall('.//frame'):
                fn = int(frame.get('num', 0))
                boxes = []
                for tgt in frame.findall('.//target'):
                    attr = tgt.find('attribute')
                    vtype = (attr.get('vehicle_type','') if attr is not None else '').lower()
                    ac = DETRAC_MAP.get(vtype)
                    if ac is None: continue
                    box = tgt.find('box')
                    if box is None: continue
                    try:
                        l = float(box.get('left',0)); t = float(box.get('top',0))
                        bw = float(box.get('width',0)); bh = float(box.get('height',0))
                    except ValueError: continue
                    if bw <= 0 or bh <= 0: continue
                    cx = max(.001, min(.999, (l+bw/2)/IMG_W))
                    cy = max(.001, min(.999, (t+bh/2)/IMG_H))
                    nw = max(.001, min(.999, bw/IMG_W))
                    nh = max(.001, min(.999, bh/IMG_H))
                    boxes.append((ac, cx, cy, nw, nh))
                if boxes: frames[fn] = boxes
            if frames: seq_ann[xp.stem] = frames

        # Find all images
        all_imgs = sorted(DETRAC_ROOT.rglob('img*.jpg')) + sorted(DETRAC_ROOT.rglob('img*.png'))
        imgs_shuffled = list(all_imgs); random.shuffle(imgs_shuffled)
        held_set = set(str(p) for p in imgs_shuffled[:2000])

        HELD_DIR.mkdir(parents=True, exist_ok=True)
        (HELD_DIR/'valid'/'images').mkdir(parents=True, exist_ok=True)
        (HELD_DIR/'valid'/'labels').mkdir(parents=True, exist_ok=True)

        nv = n_held = 0
        for img in tqdm(all_imgs, desc='UA-DETRAC'):
            seq_name = img.parent.name
            m = re.search(r'(\d+)', img.stem[-8:])
            fn = int(m.group(1)) if m else 0
            ann = seq_ann.get(seq_name, {}).get(fn)
            if not ann: continue
            lines = [f'{ac} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}' for ac,cx,cy,nw,nh in ann]
            stem = f'detrac_{seq_name}_{img.stem}'
            if str(img) in held_set:
                hi = HELD_DIR/'valid'/'images'; hl = HELD_DIR/'valid'/'labels'
                link = hi/(stem+img.suffix)
                if not link.exists(): _imglink(img, link)
                (hl/(stem+'.txt')).write_text('\n'.join(lines))
                n_held += 1
            else:
                dst = 'train' if random.random() < 0.9 else 'valid'
                di = MERGED/dst/'images'; dl = MERGED/dst/'labels'
                link = di/(stem+img.suffix)
                if not link.exists(): _imglink(img, link)
                (dl/(stem+'.txt')).write_text('\n'.join(lines))
                nv += 1

        (HELD_DIR/'data.yaml').write_text(
            f'path: {HELD_DIR.resolve()}\nval: valid/images\nnc: {NC}\nnames: {CLASSES}\n')
        print(f'UA-DETRAC: {nv:,} train/val | {n_held:,} held-out locked')
        FLAG.touch(); print('UA-DETRAC: done')
    if DETRAC_ROOT is not None and str(DETRAC_ROOT).startswith(str(WORK)):
        _shutil.rmtree(DETRAC_ROOT, ignore_errors=True)
        print(f'  Freed {DETRAC_ROOT}')

In [ ]:
# ── BDD100K → YOLO ───────────────────────────────────────────────────────────
# motorcycle ×3 oversample, bicycle ×2 oversample for class balance.
# Checks /kaggle/input/ first (attached dataset), falls back to Kaggle API.
import shutil as _shutil

def _imglink(src: 'Path', dst: 'Path'):
    # Hardlink if same filesystem (deletable raw), symlink if cross-fs (input/).
    try: os.link(src, dst)
    except OSError: os.symlink(src.resolve(), dst)
import os, subprocess, yaml
from pathlib import Path
from tqdm import tqdm

FLAG = OUT_DIR / '.bdd_done'
if FLAG.exists():
    print('BDD100K: already done — skipping')
else:
    name_to_argus = {'car':0,'automobile':0,'motorcycle':1,'motorbike':1,'motor':1,
                     'bus':2,'truck':3,'van':3,'bicycle':4,'bike':4}

    # ── Locate BDD100K — attached dataset OR API download ────────────────────
    BDD_DIR = None

    # 1. Check /kaggle/input/ for an attached dataset
    ATTACHED_SLUGS = ['bdd100k-yolo-format','bdd100k-dataset','bdd100k','bdd100k-yolo',
                      'bdd100k-yolo-v8','bdd-100k','bdd100k_yolo','bdd_100k','bdd100k_yolo_format']
    for slug in ATTACHED_SLUGS:
        candidate = INPUT / slug
        if candidate.exists() and any(candidate.rglob('*.jpg')):
            BDD_DIR = candidate
            print(f'BDD100K: found attached dataset at {BDD_DIR}')
            break

    # 2. Nested datasets/<user>/<ds>/ search (Kaggle merges user datasets this way)
    if BDD_DIR is None:
        ds_root = INPUT / 'datasets'
        if ds_root.exists():
            for user_dir in sorted(ds_root.iterdir()):
                if not user_dir.is_dir() or BDD_DIR: break
                for ds_dir in sorted(user_dir.iterdir()):
                    if not ds_dir.is_dir(): continue
                    # Check data.yaml for BDD class names (rider, traffic light, traffic sign)
                    for yf in ds_dir.rglob('data.yaml'):
                        try:
                            cfg = yaml.safe_load(yf.read_text())
                            names_list = cfg.get('names', [])
                            if isinstance(names_list, dict): names_list = list(names_list.values())
                            bdd_markers = {'rider', 'traffic light', 'traffic sign', 'motor'}
                            if bdd_markers & {n.strip().lower() for n in names_list}:
                                BDD_DIR = ds_dir
                                print(f'BDD100K: found at datasets/{user_dir.name}/{ds_dir.name}/ (BDD class names)')
                                break
                        except Exception: pass
                    if BDD_DIR: break
                    # Fallback: largest unlabelled dataset (>50k jpgs, not IDD)
                    if not BDD_DIR and 'idd' not in str(ds_dir).lower() and 'argus' not in str(ds_dir).lower() and 'weights' not in str(ds_dir).lower() and 's2' not in ds_dir.name.lower():
                        jpg_n = sum(1 for _ in ds_dir.rglob('*.jpg'))
                        if jpg_n > 50000:
                            BDD_DIR = ds_dir
                            print(f'BDD100K: found at datasets/{user_dir.name}/{ds_dir.name}/ ({jpg_n} images, assumed BDD)')
                            break

    # 3. Fallback: API download
    if BDD_DIR is None and KGL_USER:
        RAW = WORK / '_bdd_raw'; RAW.mkdir(parents=True, exist_ok=True)
        dl_flag = RAW / '.downloaded'
        API_SLUGS = ['farzadnekouei/bdd100k-yolo-format','awsaf49/bdd100k-dataset',
                     'a2zcode/bdd100k','gautamchettiar/bdd100k','a7madmostafa/bdd100k-yolo']
        if dl_flag.exists():
            print('BDD100K: already downloaded'); BDD_DIR = RAW
        else:
            for slug in API_SLUGS:
                r = subprocess.run(['kaggle','datasets','download','-d',slug,
                                    '-p',str(RAW),'--unzip'], capture_output=True, text=True)
                if r.returncode == 0:
                    dl_flag.touch(); BDD_DIR = RAW
                    print(f'  Downloaded via {slug}'); break
                print(f'  {slug}: {r.stderr.strip()[:120]}')
            if BDD_DIR is None:
                print('  All BDD100K slugs failed — skipping')

    if BDD_DIR is None:
        print('BDD100K: not found — attach dataset or set Kaggle Secrets')
    else:
        # Auto-detect class map from data.yaml
        bdd_map = None
        for yf in sorted(BDD_DIR.rglob('*.yaml')):
            try:
                cfg = yaml.safe_load(yf.read_text())
                names = cfg.get('names', [])
                if isinstance(names, dict): names = [names[k] for k in sorted(names)]
                m = {i: name_to_argus[n.lower().strip()] for i, n in enumerate(names)
                     if n.lower().strip() in name_to_argus}
                if m: bdd_map = m; print(f'  BDD map (from {yf.name}): {m}'); break
            except Exception: pass
        if bdd_map is None:
            # BDD100K standard 10-class ordering:
            # pedestrian(0) rider(1) car(2) truck(3) bus(4) train(5)
            # motorcycle(6) bicycle(7) traffic light(8) traffic sign(9)
            bdd_map = {2:0, 6:1, 4:2, 3:3, 7:4}
            print(f'  BDD map: fallback {bdd_map}')

        for split, dst in [('train','train'), ('val','valid')]:
            idirs = [d for d in BDD_DIR.rglob('images') if split in str(d)]
            ldirs = [d for d in BDD_DIR.rglob('labels') if split in str(d)]
            if not idirs: print(f'  BDD {split}: not found — skip'); continue
            idir = idirs[0]
            ldir = ldirs[0] if ldirs else idir.parent.parent/'labels'/split
            di = MERGED/dst/'images'; dl = MERGED/dst/'labels'
            nv = nm = nb_cnt = 0
            for img in tqdm(list(idir.glob('*.*')), desc=f'BDD {split}'):
                lp = ldir / (img.stem + '.txt')
                if not lp.exists(): continue
                lines, has_m, has_b = [], False, False
                for row in lp.read_text().strip().splitlines():
                    p = row.split()
                    if not p: continue
                    try: orig = int(p[0])
                    except ValueError: continue
                    if orig in bdd_map:
                        ac = bdd_map[orig]; lines.append(f'{ac} ' + ' '.join(p[1:]))
                        if ac == 1: has_m = True
                        if ac == 4: has_b = True
                if not lines: continue
                stem = f'bdd_{img.stem}'
                link = di / (stem + img.suffix)
                if not link.exists(): _imglink(img, link)
                (dl / (stem + '.txt')).write_text('\n'.join(lines))
                nv += 1
                if has_m:
                    nm += 1
                    for k in range(2):
                        s2 = f'bdd_{img.stem}_m{k}'
                        ml = di / (s2 + img.suffix)
                        if not ml.exists(): _imglink(img, ml)
                        (dl / (s2 + '.txt')).write_text('\n'.join(lines))
                if has_b:
                    nb_cnt += 1
                    bl = di / (f'bdd_{img.stem}_b0' + img.suffix)
                    if not bl.exists(): _imglink(img, bl)
                    (dl / (f'bdd_{img.stem}_b0.txt')).write_text('\n'.join(lines))
            print(f'  BDD {split}: {nv} imgs | moto×3: {nm} | bike×2: {nb_cnt}')
        FLAG.touch(); print('BDD100K: done')
        # Free downloaded raw — hardlinks in MERGED are intact
        if str(BDD_DIR).startswith(str(WORK)):
            _shutil.rmtree(BDD_DIR, ignore_errors=True)
            print(f'  Freed {BDD_DIR}')

In [ ]:
# ── IDD (Indian Driving Dataset) → YOLO ──────────────────────────────────────
# motorcycle/autorickshaw x4, bicycle x2 oversample.
import xml.etree.ElementTree as ET
import os, random, yaml, subprocess
from pathlib import Path
from tqdm import tqdm

FLAG = OUT_DIR / '.idd_done'
if FLAG.exists():
    print('IDD: already done -- skipping')
else:
    IDD_NAME_MAP = {
        'car':0, 'auto':0, 'automobile':0,
        'autorickshaw':1, 'auto rickshaw':1, 'three wheeler':1,
        'motorcycle':1, 'motorbike':1, 'two wheeler':1, 'scooter':1,
        'bus':2,
        'truck':3, 'van':3, 'tempo':3, 'vehicle_fallback':3,
        'bicycle':4, 'cycle':4,
    }

    IDD_ROOT = None

    # ── 1. Known slugs (Kaggle mounts abhishekprajapat/idd-20k as idd-20k or idd_20k) ──
    for slug in ['idd-20k', 'idd_20k', 'idd-dataset-yolo-format', 'idd', 'idd-detection', 'idd20k', 'idd-yolo']:
        if (INPUT / slug).exists():
            IDD_ROOT = INPUT / slug
            print(f'IDD: found at /kaggle/input/{slug}/')
            break

    # ── 2. Search for idd20k_final by name (handles datasets/ merged folder) ──
    # When Kaggle combines user datasets under /kaggle/input/datasets/<user>/,
    # IDD_20K is at datasets/<user>/idd-20k/idd20k_final/
    if IDD_ROOT is None:
        hit = next(INPUT.rglob('idd20k_final'), None)
        if hit and hit.is_dir():
            IDD_ROOT = hit
            print(f'IDD: found idd20k_final at {hit}')

    # ── 3. Search all data.yaml files for IDD-specific class names ─────────────
    if IDD_ROOT is None:
        for yf in sorted(INPUT.rglob('data.yaml')):
            try:
                cfg = yaml.safe_load(yf.read_text())
                names_list = cfg.get('names', [])
                if isinstance(names_list, dict): names_list = list(names_list.values())
                idd_markers = {'autorickshaw', 'two wheeler', 'scooter', 'tempo', 'cycle'}
                if idd_markers & {n.strip().lower() for n in names_list}:
                    IDD_ROOT = yf.parent
                    print(f'IDD: found via data.yaml at {yf}')
                    break
            except Exception:
                pass

    # ── 4. Search for VOC Annotations/ dir ────────────────────────────────────
    if IDD_ROOT is None:
        for ann in INPUT.rglob('Annotations'):
            if ann.is_dir() and next(ann.glob('*.xml'), None):
                IDD_ROOT = ann.parent
                print(f'IDD: found VOC dataset at {IDD_ROOT}')
                break

    # ── 5. API download ────────────────────────────────────────────────────────
    if IDD_ROOT is None and KGL_USER:
        IDD_DL = WORK / '_idd_raw'
        IDD_DL.mkdir(parents=True, exist_ok=True)
        for slug in ['abhishekprajapat/idd-20k', 'ashwinak/idd-dataset-yolo-format']:
            r = subprocess.run(['kaggle', 'datasets', 'download', '-d', slug,
                                '-p', str(IDD_DL), '--unzip'],
                               capture_output=True, text=True)
            if r.returncode == 0:
                IDD_ROOT = IDD_DL
                print(f'IDD: downloaded via {slug}')
                break
            print(f'  {slug}: {r.stderr.strip()[:100]}')

    if IDD_ROOT is None:
        print('IDD: not found')
        mounted = [d.name for d in INPUT.iterdir() if d.is_dir()] if INPUT.exists() else []
        print(f'  Mounted top-level: {mounted}')
        print('  Attach abhishekprajapat/idd-20k as a Kaggle dataset and rerun.')
    else:
        # ── Detect format ─────────────────────────────────────────────────────
        yolo_yamls = list(IDD_ROOT.rglob('data.yaml'))
        yolo_idirs = list(IDD_ROOT.rglob('images'))
        yolo_ldirs = list(IDD_ROOT.rglob('labels'))
        voc_ann    = next(IDD_ROOT.rglob('Annotations'), None)
        voc_img    = next(IDD_ROOT.rglob('JPEGImages'), None)

        # Also check for images directly in train/ valid/ (no images/ subdir)
        direct_splits = []
        for split_kw, dst in [('train', 'train'), ('val', 'valid'), ('valid', 'valid')]:
            sdir = IDD_ROOT / split_kw
            if sdir.exists():
                imgs = list(sdir.glob('*.jpg')) + list(sdir.glob('*.png')) + list(sdir.glob('*.jpeg'))
                lbls = list(sdir.glob('*.txt'))
                if imgs and lbls:
                    direct_splits.append((dst, sdir, sdir))

        if yolo_idirs and yolo_ldirs:
            print('IDD: YOLO format (images/ + labels/ subdirs)')
            idd_map = {}
            if yolo_yamls:
                try:
                    cfg = yaml.safe_load(yolo_yamls[0].read_text())
                    names = cfg.get('names', [])
                    if isinstance(names, dict): names = [names[k] for k in sorted(names)]
                    idd_map = {i: IDD_NAME_MAP.get(n.strip().lower())
                               for i, n in enumerate(names)}
                    idd_map = {k: v for k, v in idd_map.items() if v is not None}
                    print(f'  class map (from {yolo_yamls[0].name}): {idd_map}')
                except Exception as e:
                    print(f'  data.yaml parse failed: {e}')
            if not idd_map:
                # Sample first label file to see what class IDs exist
                sample_lbl = next(IDD_ROOT.rglob('*.txt'), None)
                if sample_lbl:
                    ids = set()
                    for row in sample_lbl.read_text().strip().splitlines()[:20]:
                        p = row.split()
                        if p:
                            try: ids.add(int(p[0]))
                            except ValueError: pass
                    print(f'  No data.yaml -- sample class IDs from {sample_lbl.name}: {sorted(ids)}')
                    # Build map: anything in 0-4 stays, use IDD_NAME_MAP for remapping
                    idd_map = {i: i for i in range(10) if i <= 4}
                    print(f'  class map: identity 0-4 (set data.yaml for correct mapping)')

            for kw, dst in [('train', 'train'), ('val', 'valid'), ('test', 'valid')]:
                idirs = [d for d in yolo_idirs if kw in str(d).lower()]
                ldirs = [d for d in yolo_ldirs if kw in str(d).lower()]
                if not idirs: continue
                idir = idirs[0]
                # Try 3 label dir patterns: from rglob, labels/<split>/, <split>/labels/
                if ldirs:
                    ldir = ldirs[0]
                elif (IDD_ROOT / 'labels' / kw).exists():
                    ldir = IDD_ROOT / 'labels' / kw
                else:
                    ldir = idir.parent / 'labels'
                print(f'  IDD {kw}: images={idir.relative_to(INPUT)} labels={ldir.relative_to(INPUT) if ldir.exists() else ldir}')
                di = MERGED / dst / 'images'
                dl = MERGED / dst / 'labels'
                nv = nm = nb = 0
                for img in tqdm(list(idir.glob('*.*')), desc=f'IDD {kw}'):
                    lp = ldir / (img.stem + '.txt')
                    if not lp.exists(): continue
                    lines, has_m, has_b = [], False, False
                    for row in lp.read_text().strip().splitlines():
                        p = row.split()
                        if not p: continue
                        try: orig = int(p[0])
                        except ValueError: continue
                        if orig in idd_map:
                            ac = idd_map[orig]
                            lines.append(f'{ac} ' + ' '.join(p[1:]))
                            if ac == 1: has_m = True
                            if ac == 4: has_b = True
                    if not lines: continue
                    stem = f'idd_{kw}_{img.stem}'
                    link = di / (stem + img.suffix)
                    if not link.exists(): os.symlink(img.resolve(), link)
                    (dl / (stem + '.txt')).write_text('\n'.join(lines))
                    nv += 1
                    if has_m:
                        nm += 1
                        for k in range(3):  # x4 total
                            mlink = di / (f'idd_{kw}_{img.stem}_m{k}' + img.suffix)
                            if not mlink.exists(): os.symlink(img.resolve(), mlink)
                            (dl / (f'idd_{kw}_{img.stem}_m{k}.txt')).write_text('\n'.join(lines))
                    if has_b:
                        nb += 1   # x2 total
                        blink = di / (f'idd_{kw}_{img.stem}_b0' + img.suffix)
                        if not blink.exists(): os.symlink(img.resolve(), blink)
                        (dl / (f'idd_{kw}_{img.stem}_b0.txt')).write_text('\n'.join(lines))
                print(f'  IDD {kw}: {nv} imgs | moto x4: {nm} | bike x2: {nb}')
                if nv == 0 and list(idir.glob('*.*')):
                    sample = next(idir.glob('*.*'))
                    slp = ldir / (sample.stem + '.txt')
                    print(f'  DEBUG: sample img={sample.name} label_exists={slp.exists()}')
                    if slp.exists():
                        print(f'  DEBUG: label content: {slp.read_text()[:200]}')

        elif direct_splits:
            print('IDD: YOLO format (images directly in train/val dirs)')
            idd_map = {i: i for i in range(10) if i <= 4}
            for dst, idir, ldir in direct_splits:
                di = MERGED / dst / 'images'
                dl = MERGED / dst / 'labels'
                nv = nm = nb = 0
                for img in tqdm(list(idir.glob('*.jpg')) + list(idir.glob('*.png')),
                                desc=f'IDD direct {dst}'):
                    lp = ldir / (img.stem + '.txt')
                    if not lp.exists(): continue
                    lines, has_m, has_b = [], False, False
                    for row in lp.read_text().strip().splitlines():
                        p = row.split()
                        if not p: continue
                        try: orig = int(p[0])
                        except ValueError: continue
                        if orig in idd_map:
                            ac = idd_map[orig]
                            lines.append(f'{ac} ' + ' '.join(p[1:]))
                            if ac == 1: has_m = True
                            if ac == 4: has_b = True
                    if not lines: continue
                    stem = f'idd_{dst}_{img.stem}'
                    link = di / (stem + img.suffix)
                    if not link.exists(): os.symlink(img.resolve(), link)
                    (dl / (stem + '.txt')).write_text('\n'.join(lines))
                    nv += 1
                    if has_m:
                        nm += 1
                        for k in range(3):
                            mlink = di / (f'idd_{dst}_{img.stem}_m{k}' + img.suffix)
                            if not mlink.exists(): os.symlink(img.resolve(), mlink)
                            (dl / (f'idd_{dst}_{img.stem}_m{k}.txt')).write_text('\n'.join(lines))
                    if has_b:
                        nb += 1
                        blink = di / (f'idd_{dst}_{img.stem}_b0' + img.suffix)
                        if not blink.exists(): os.symlink(img.resolve(), blink)
                        (dl / (f'idd_{dst}_{img.stem}_b0.txt')).write_text('\n'.join(lines))
                print(f'  IDD direct {dst}: {nv} imgs | moto x4: {nm} | bike x2: {nb}')

        elif voc_ann and voc_img:
            print('IDD: VOC XML format')
            xmls = list(voc_ann.glob('*.xml'))
            random.seed(42); random.shuffle(xmls)
            idx = int(len(xmls) * 0.9)
            for dst, xlist in [('train', xmls[:idx]), ('valid', xmls[idx:])]:
                di = MERGED / dst / 'images'
                dl = MERGED / dst / 'labels'
                nv = nm = nb = 0
                for xp in tqdm(xlist, desc=f'IDD VOC {dst}'):
                    try: root_el = ET.parse(xp).getroot()
                    except ET.ParseError: continue
                    sz = root_el.find('size')
                    if sz is None or sz.find('width') is None: continue
                    W, H = int(sz.find('width').text), int(sz.find('height').text)
                    if W <= 0 or H <= 0: continue
                    lines, has_m, has_b = [], False, False
                    for obj in root_el.findall('object'):
                        nm_raw = obj.find('name').text.strip()
                        ac = IDD_NAME_MAP.get(nm_raw.lower())
                        if ac is None: continue
                        bb = obj.find('bndbox')
                        x1, y1, x2, y2 = (float(bb.find(t).text)
                                           for t in ['xmin', 'ymin', 'xmax', 'ymax'])
                        lines.append(
                            f'{ac} {max(.001, min(.999, (x1+x2)/2/W)):.6f}'
                            f' {max(.001, min(.999, (y1+y2)/2/H)):.6f}'
                            f' {max(.001, min(.999, (x2-x1)/W)):.6f}'
                            f' {max(.001, min(.999, (y2-y1)/H)):.6f}')
                        if ac == 1: has_m = True
                        if ac == 4: has_b = True
                    if not lines: continue
                    ip = next((voc_img / (xp.stem + e) for e in ['.jpg', '.jpeg', '.png']
                               if (voc_img / (xp.stem + e)).exists()), None)
                    if ip is None: continue
                    stem = f'idd_{xp.stem}'
                    link = di / (stem + ip.suffix)
                    if not link.exists(): os.symlink(ip.resolve(), link)
                    (dl / (stem + '.txt')).write_text('\n'.join(lines))
                    nv += 1
                    if has_m:
                        nm += 1
                        for k in range(3):
                            mlink = di / (f'idd_{xp.stem}_m{k}' + ip.suffix)
                            if not mlink.exists(): os.symlink(ip.resolve(), mlink)
                            (dl / (f'idd_{xp.stem}_m{k}.txt')).write_text('\n'.join(lines))
                    if has_b:
                        nb += 1
                        blink = di / (f'idd_{xp.stem}_b0' + ip.suffix)
                        if not blink.exists(): os.symlink(ip.resolve(), blink)
                        (dl / (f'idd_{xp.stem}_b0.txt')).write_text('\n'.join(lines))
                print(f'  IDD VOC {dst}: {nv} | moto x4: {nm} | bike x2: {nb}')
        else:
            print(f'IDD: unrecognised structure at {IDD_ROOT}')
            print(f'  rglob images/: {yolo_idirs[:3]}')
            print(f'  rglob labels/: {yolo_ldirs[:3]}')
            print(f'  direct splits: {direct_splits}')
        FLAG.touch()
        print('IDD: done')

In [ ]:
# ── Dataset stats + data.yaml ────────────────────────────────────────────────
import shutil
from pathlib import Path
from collections import Counter

def _count(split):
    ldir = MERGED/split/'labels'
    imgs = list((MERGED/split/'images').glob('*.*'))
    stats = Counter()
    for lp in ldir.glob('*.txt'):
        for row in lp.read_text().splitlines():
            p = row.strip().split()
            if p:
                try: stats[int(p[0])] += 1
                except ValueError: pass
    return imgs, stats

print('='*60)
for split in ['train','valid']:
    imgs, stats = _count(split)
    total = sum(stats.values())
    bad = {k:v for k,v in stats.items() if k<0 or k>=NC}
    print(f'\n{split}: {len(imgs):,} images | {total:,} boxes')
    for cid, name in enumerate(CLASSES):
        bar = '█' * min(40, int(40*stats.get(cid,0)/max(total,1)))
        print(f'  {cid} {name:12s}: {stats.get(cid,0):8,}  {bar}')
    if bad:
        print(f'  ⚠  OUT-OF-RANGE IDs: {bad}')
    else:
        print(f'  ✓ all IDs in [0,{NC-1}]')
print('='*60)

# Per-source breakdown
print('\nSource breakdown (train images):')
sources = Counter()
for p in (MERGED/'train'/'images').glob('*.*'):
    prefix = p.stem.split('_')[0]
    sources[prefix] += 1
for src, n in sorted(sources.items(), key=lambda x: -x[1]):
    print(f'  {src:12s}: {n:,}')

# Disk usage
total_gb, used_gb, free_gb = (x/1024**3 for x in shutil.disk_usage('/kaggle/working'))
print(f'\nDisk /kaggle/working: {used_gb:.1f} GB used / {total_gb:.1f} GB total ({free_gb:.1f} GB free)')

n_train = len(list((MERGED/'train'/'images').glob('*.*')))
if n_train < 500:
    raise AssertionError(
        f'Only {n_train} training images found.\n'
        'BDD100K and UA-DETRAC likely failed to load.\n'
        'Fix: set KAGGLE_USERNAME + KAGGLE_KEY in Secrets, OR attach BDD100K as a dataset.')
elif n_train < 5_000:
    print(f'\n⚠  Only {n_train} images — training will run but accuracy will be limited.')
    print('   Add BDD100K (attach dataset or set Kaggle Secrets) for full training.')
else:
    print(f'\n✓ {n_train:,} training images ready')

yaml_path.write_text(
    f'path: {MERGED.resolve()}\ntrain: train/images\nval:   valid/images\n'
    f'\nnc: {NC}\nnames: {CLASSES}\n')
print(f'data.yaml → {yaml_path}')

In [ ]:
# ── Phase B resume — continues from epoch 4 to epoch 20 ──────────────────────
# YOLO reads args.yaml from PHB_DIR to restore the full training config:
#   lr0=2e-4, cosine decay, close_mosaic=8, box=9.0, cls=0.3, dfl=2.0
# close_mosaic=8 on a 20-epoch run → mosaic turns OFF at epoch 13.
# This session reaches epochs 5-7 before the 9.8h session limit hits.
# Restart this notebook after each timeout — it auto-resumes from last.pt.
from ultralytics import YOLO
import time

assert PHB_LAST.exists(), (
    'last.pt not found — run checkpoint-setup cell first')
assert yaml_path.exists(), (
    'data.yaml not found — run dataset cells first')

# Patch args.yaml data path (must point to current session's data.yaml)
args_file = PHB_DIR / 'args.yaml'
if args_file.exists():
    txt = args_file.read_text()
    if str(yaml_path) not in txt:
        txt = txt.replace(
            'data: /kaggle/working/argus_data/merged/data.yaml',
            f'data: {yaml_path}')
        args_file.write_text(txt)
        print(f'Patched args.yaml data path → {yaml_path}')

print(f'Resuming Phase B from {PHB_LAST}')
print(f'Expected: epochs 5-7 this session, 8-9 next session')
print(f'Mosaic OFF from epoch 13 (close_mosaic=8)')
print()

t0 = time.time()
model = YOLO(str(PHB_LAST))
model.train(resume=True)

elapsed = (time.time()-t0)/3600
print(f'\nSession done in {elapsed:.1f} h')

# Save best as argus_s2_cont.pt for next stage
import shutil
best = PHB_BEST if PHB_BEST.exists() else PHB_LAST
shutil.copy2(best, WORK/'argus_s2_cont.pt')
print(f'\n✓ Best checkpoint → {WORK}/argus_s2_cont.pt')
print('  Download from Output tab after each session')
print('  Final output after epoch 9: upload as argus_s2.pt for S2.5')


In [ ]:
# ── Quick validation after session (run manually after training cell) ─────────
from ultralytics import YOLO

candidates = [WORK/'argus_s2_cont.pt', PHB_BEST, PHB_LAST]
EVAL = next((c for c in candidates if c.exists()), None)
assert EVAL, 'No checkpoint found'
print(f'Evaluating: {EVAL}')

model = YOLO(str(EVAL))
m = model.val(data=str(yaml_path), imgsz=IMGSZ, batch=BATCH,
              device=DEVICE, conf=0.001, iou=0.6, verbose=True)

print(f'\n  mAP50    : {m.box.map50:.4f}')
print(f'  mAP50-95 : {m.box.map:.4f}  <- PRIMARY')
for name, ap50, ap in zip(CLASSES, m.box.ap50, m.box.ap):
    print(f'  {name:12s}: AP50={ap50:.3f}  AP50-95={ap:.3f}')
